# 低比特量化

> 想在一张 16 GB 的显卡上跑 14B 模型：BF16 权重 28 GB，差得远；把每个参数压到 4 bit，只剩 7 GB，放得下了。这笔「4 倍缩水」的账很好算，真正的问题是——数变少了这么多，模型为什么还能用？用到什么程度会坏？
>
> 上一章的结论是 Decode 的瓶颈在数据搬运：权重每一步都要完整读一遍。量化直接把搬运量变小了，所以它同时优化显存和 TPOT。
>
> 本章介绍量化的四组核心内容：
>
> 1. **INT4 的基本原理**：16 个格子怎么装下浮点数——scale 与粒度。
> 2. **整数与浮点格式**：INT4 / INT8 之外，FP8 与 FP4 为什么存在。
> 3. **算法与记号**：GPTQ、AWQ、SmoothQuant 各自在救什么，`W4A16` / `W8A8` 在读什么。
> 4. **生态与实战**：常见格式怎么亲手做一版、按部署目标怎么选，量化的收益与代价。

先从最直观的一笔账开始。

## 1. 模型大小与精度

推理时的显存大头有两块：权重（固定大小，Decode 每步都要搬）和 KV Cache（随上下文增长）。量化最开始瞄准的是权重——它既是显存大头，又是上一章说的「每步搬运对象」。先看看不同精度下权重到底占多大地方：

In [ ]:
def weight_size_gb(params_billion, bits):
    """按参数量和每参数比特数估算权重显存（GB）"""
    return params_billion * 1e9 * bits / 8 / 1e9

for size in [7, 70]:
    for bits in [16, 8, 4]:
        print(f"{size:>2d}B @ {bits:2d}-bit -> {weight_size_gb(size, bits):6.1f} GB")

print()
print("关键观察：70B BF16 约 140 GB，单卡放不下；压到 4-bit 约 35 GB，一张卡就装下了")

In [ ]:
# 显存地图：参数量 × 精度决定每个 Decode step 要搬多少权重
import matplotlib.pyplot as plt

bits_options = [16, 8, 4]
sizes = [7, 70]
xs = range(len(bits_options))
width = 0.35

plt.figure(figsize=(6.5, 3.5))
for j, size in enumerate(sizes):
    values = [weight_size_gb(size, b) for b in bits_options]
    plt.bar([x + (j - 0.5) * width for x in xs], values, width=width,
            label=f"{size}B params")
plt.xticks(list(xs), [f"{b}-bit" for b in bits_options])
plt.ylabel("weight size (GB)")
plt.title("Smaller dtype = less memory to load every decode step")
plt.legend()
plt.show()

这张图回答了「为什么量化值得做」，但紧接着就是一个更根本的问题：16 bit 浮点有 6 万多种取值，4 bit 整数只有 16 种——数变少了这么多，模型为什么还没坏？要回答它，得先弄清楚「量化」这个词到底指什么操作。

## 2. 从浮点到 INT4

INT4 的意思是每个数用 4 个 bit 表示，一共只有 16 个不同的取值（对称量化常用 -7 到 +7）。而一段权重里有几十亿个不同的浮点数，要把它们装进 16 个格子——所以量化不是「把小数四舍五入成整数」，而是「把一段连续的取值范围，映射到 16 个固定的档位上」。

档位之间的间隔就是 **scale**。对称量化可以写成：

$$
q = \mathrm{clip}(\mathrm{round}(x/s),\ q_{min},\ q_{max}),
\qquad \hat{x} = s \cdot q
$$

先手算两个数再交给代码。取 7 个数里的 `-0.72` 和 `0.63`，`scale = 1.0 / 7 ≈ 0.143`：

```text
-0.72 / 0.143 ≈ -5.04  ->  round 成 -5  ->  反量化 -5 × 0.143 ≈ -0.714，误差约 0.006
 0.63 / 0.143 ≈  4.41  ->  round 成  4  ->  反量化  4 × 0.143 ≈  0.571，误差约 0.059
```

每个数都被拉到最近的档位上，误差最多半个格子（scale / 2）。单个数看误差很小，但整份权重有几百万个数，累计误差就是模型掉分的来源——后面所有量化算法，都是在想办法让这笔累计误差更小。

In [ ]:
import numpy as np

x = np.array([-1.0, -0.72, -0.31, 0.0, 0.18, 0.63, 1.0], dtype=np.float32)
qmax = 7
scale = np.max(np.abs(x)) / qmax
q = np.round(x / scale).clip(-qmax, qmax).astype(np.int32)
x_hat = q * scale

print("scale:", round(float(scale), 4))
print("float:", x)
print("INT4 :", q)
print("反量化:", np.round(x_hat, 3))
print()
print(f"关键观察：最大误差 {np.max(np.abs(x - x_hat)):.4f} < scale/2 = {scale/2:.4f}")
print("每个数都被拉到最近的格子——量化误差就是这几百万次拉扯的累加")

## 3. 量化粒度

全层共用一个 scale，隐含着一个假设：所有数值的范围差不多。真实权重并不配合——不同 channel 的范围经常差好几倍，个别 outlier 通道一旦出现，整层共用的 scale 就被撑大，其他通道只能挤在几个粗糙的格子里。

改进方向很自然：scale 分细一点。

```text
per-tensor  : 整个张量一个 scale
per-channel : 每个 channel 一个 scale（outlier 只污染自己）
per-group   : 每 128 个数一个 scale（现代 4-bit 方案的默认粒度）
```

粒度越细，误差越小，代价是 scale 元数据变多、Kernel 实现更复杂。用一份带 outlier 的模拟权重，把三种粒度各跑一遍：

In [ ]:
np.random.seed(7)
W = np.random.randn(4, 16).astype(np.float32) * 0.25
W[1] *= 8  # 第 1 个 channel 是 outlier，范围被拉大 8 倍

def qdq_tensor(a):
    """整层共用一个 scale"""
    s = max(np.max(np.abs(a)) / 7, 1e-12)
    q = np.round(a / s).clip(-7, 7)
    return q * s

def qdq_channel(a):
    """每个 channel 一个 scale（按行）"""
    s = np.maximum(np.max(np.abs(a), axis=1, keepdims=True) / 7, 1e-12)
    q = np.round(a / s).clip(-7, 7)
    return q * s

def qdq_group(a, group=4):
    """每 4 个相邻元素一组，各用各的 scale"""
    out = np.zeros_like(a)
    for r in range(a.shape[0]):
        for c0 in range(0, a.shape[1], group):
            block = a[r, c0:c0 + group]
            s = max(np.max(np.abs(block)) / 7, 1e-12)
            out[r, c0:c0 + group] = np.round(block / s).clip(-7, 7) * s
    return out

errs = {}
for name, fn in [("per-tensor", qdq_tensor), ("per-channel", qdq_channel),
                 ("per-group(4)", lambda a: qdq_group(a, 4))]:
    errs[name] = float(np.mean(np.abs(W - fn(W))))
    print(f"{name:<14} MAE = {errs[name]:.4f}")

print()
print("关键观察：outlier 通道存在时，粒度越细，其他通道被连累得越轻")

In [ ]:
# 同一份权重、三种粒度的误差对比：越绿误差越小
import matplotlib.pyplot as plt

plt.figure(figsize=(5.5, 3.2))
plt.bar(errs.keys(), errs.values(), color=["tab:red", "tab:orange", "tab:green"])
plt.ylabel("MAE (lower is better)")
plt.title("Finer granularity -> smaller quantization error")
plt.show()

## 4. 权重与 Activation

到目前为止压的都是权重。但模型前向里还有另一个角色：**Activation**——每一层的输入，随 Prompt 内容变化。它要不要一起压，是记号里的第二个字母：

```text
W4A16 = Weight 4-bit, Activation 16-bit   <- 只压权重
W8A8  = Weight 8-bit, Activation 8-bit    <- 两者都压
W4A8  = Weight 4-bit, Activation 8-bit
```

为什么主流方案先从 weight-only 开始？因为两者难度完全不同。权重在推理时固定不变，可以离线慢慢量化、校准、补偿；activation 随每个输入变化，没有「离线」的机会，而且经常在固定的少数通道上出现大幅 outlier——同一套格子很难装下所有输入。

这个差异正是下一节三个算法的出发点：它们各自救的是「量化误差」的不同侧面。

## 5. GPTQ、AWQ 与 SmoothQuant

先把基准说清楚：直接 round 到最近格子（RTN，Round-To-Nearest）就是第 2 节做的事，也是所有量化算法的 baseline。8-bit 下 RTN 已经够用；到 4-bit，RTN 的累计误差开始伤模型——三个算法都在想办法把这个误差救回来，思路各不相同。

**GPTQ：量化完的误差，能不能补回来？** 逐列量化权重，每量化一列就用二阶信息（近似 Hessian）估计这列的误差对输出的影响，把误差分摊到还没量化的列上去补偿。效果是「量化后的模型输出」尽量贴近「量化前的模型输出」。它是 post-training、weight-only 方法的代表。

**AWQ：所有通道同样重要吗？** 不是。观察 activation 的统计会发现，少数通道的数值特别大——这些通道上的量化误差会被放大，伤模型最重。AWQ 按 activation 的分布给重要通道更精细的 scale（等价于放大它们再量化），不需要反向传播，校准成本比 GPTQ 低。

**SmoothQuant：activation 难量化，能不能把难度挪走？** activation 的 outlier 集中在固定通道，权重却是好量化的。SmoothQuant 做一个等价变换：把每个通道的 activation 除以一个平滑因子，同时把对应权重乘上这个因子——数学上输出不变，但数值难度从 activation 一侧挪到了 weight 一侧，两边终于都「可量化」了。它是 W8A8 路线的代表。

把三个名字放回各自的位置：

| 算法 | 救的问题 | 路线 |
|:---|:---|:---|
| GPTQ | 4-bit 权重的累计误差 | weight-only，二阶补偿 |
| AWQ | 重要通道被量化伤到 | weight-only，保护 salient channels |
| SmoothQuant | activation outlier 难量化 | W8A8，难度迁移 |

它们是**量化算法**，不是格式——同一个「4-bit」可以由 RTN、GPTQ 或 AWQ 产出，质量差一截。

## 6. PTQ、QAT 与 KV Cache 量化

两个常见的分类问题，先把词归位。

**什么时候做？** PTQ（Post-Training Quantization）是训练完再量化——本章讲的全部属于这类：拿现成 checkpoint、跑一点校准数据、产出低比特版本，不需要训练资源，这是它能普及的原因。QAT（Quantization-Aware Training）是训练（或微调）过程中就插入「量化-反量化」的模拟，让模型在损失函数里直接适应低比特误差，效果上限更高，但要付出完整的训练成本。生产里最常见的路径是：PTQ 先试，评测掉分太多再考虑 QAT。

**KV Cache 量化压的是谁？** 权重之外，「LLM 推理的计算与显存开销」一本算过：长上下文、多并发下 KV Cache 会长成显存大户——把 KV 从 FP16 压到 FP8 或 INT8，等于把那笔账再砍一半。它和 `W4A16` 压的是不同对象，引擎配置里常写成 `kv_cache_dtype=fp8` 这类参数。

到这里，格子都是整数档位。还有一整族量化走另一条路——浮点格式。

## 7. 浮点格式 FP8 与 FP4

INT4 / INT8 的格子是**均匀**的：相邻档位间隔相同。但真实权重的分布是「大量数值挤在 0 附近，少数 outlier 伸得很远」——均匀格子对这种形状两头浪费：0 附近的数值挤成一团分不清档，远处的稀疏区间又占着大片没人用的范围。

浮点格式的想法是让格子**不均匀**：数值用「指数 + 尾数」表示，靠近 0 的区域格子密，远离 0 的区域格子稀——正好贴合「中间密、两头稀」的分布形状。

$$
\underbrace{\pm}_{符号}\ \underbrace{2^{e}}_{指数位}\ \underbrace{\times 1.m}_{尾数位}
$$

FP16、BF16、FP8、FP4 都是这套结构的不同位宽组合：

| 格式 | 比特 | 指数 / 尾数 | 出现场景 |
|:---|:---|:---|:---|
| FP16 | 16 | 5 / 10 | 传统混合精度训练 |
| BF16 | 16 | 8 / 7 | 现代训练默认，范围大 |
| FP8 E4M3 | 8 | 4 / 3 | 精度优先的推理 / 训练 |
| FP8 E5M2 | 8 | 5 / 2 | 范围优先，能装下更大的数值 |
| FP4 E2M1 | 4 | 2 / 1 | 最新一代硬件 |

E4M3 和 E5M2 的区别就是这个取舍：指数位多一位，能表示的范围更大，但每段区间里的档位变稀。

硬件是这波格式的真正推手：Hopper 一代 GPU（H100 / H200）原生支持 FP8 矩阵乘，Blackwell 一代进一步支持 FP4。DeepSeek-V3 从训练起就用 FP8，开源模型直接发布 FP8 checkpoint 已经是常态；FP4 生态（NVFP4、MXFP4——在 FP4 之上给每 16 / 32 个元素配一个 FP8 的 block scale）正随新硬件铺开。

浮点格子到底比整数格子好在哪？不用背结论，做个实验：同一份正态分布的权重，分别用 INT4（均匀 16 档）和 FP4 E2M1（不均匀 16 档）量化重构，比误差。

In [ ]:
# FP4 E2M1 的 16 个档位：指数让 0 附近更密
# 正档位 {0, 0.5, 1, 1.5, 2, 3, 4, 6}，对称取负数
fp4_grid = np.array([-6, -4, -3, -2, -1.5, -1, -0.5,
                     0, 0.5, 1, 1.5, 2, 3, 4, 6])

def qdq_fp4(a):
    """量化到 E2M1 的浮点档位再还原"""
    s = np.max(np.abs(a)) / 6.0
    z = a / s
    idx = np.abs(z[..., None] - fp4_grid).argmin(axis=-1)
    return fp4_grid[idx] * s

def qdq_int4(a):
    """对照：对称 INT4，均匀 15 档（-7 到 +7，惯例对称不用 -8）"""
    s = np.max(np.abs(a)) / 7.0
    return np.round(a / s).clip(-7, 7) * s

np.random.seed(0)
W = np.random.randn(1000, 100).astype(np.float32) * 0.5

err_int4 = float(np.mean(np.abs(W - qdq_int4(W))))
err_fp4 = float(np.mean(np.abs(W - qdq_fp4(W))))
print(f"INT4 (uniform 15 levels)   MAE = {err_int4:.4f}")
print(f"FP4 E2M1 (float 16 levels) MAE = {err_fp4:.4f}")
print()
print("关键观察：同样 4 bit，浮点档位对「挤在 0 附近」的分布误差更小")
print("——这就是 FP8 / FP4 在新硬件上立足的底层原因")

In [ ]:
# 左：权重直方图（中间密两头稀）；右：两种格式的误差对比
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(W.flatten(), bins=80, color="tab:gray", alpha=0.7, density=True)
axes[0].set_title("Weight distribution: dense near zero")
axes[0].set_xlabel("value")
axes[0].set_ylabel("density")

axes[1].bar(["INT4\n(uniform)", "FP4 E2M1\n(float)"], [err_int4, err_fp4],
            color=["tab:red", "tab:green"])
axes[1].set_ylabel("MAE (lower is better)")
axes[1].set_title("Same 4 bits, different grid shape")
plt.tight_layout()
plt.show()

## 8. GGUF 与 llama.cpp 生态

到这里下载模型都是 Hugging Face 的目录形态：safetensors 分片、config、tokenizer 一堆文件。另一条生态走单文件路线：**GGUF**（GPT-Generated Unified Format）把权重、词表、配置全部打包进一个文件，拷走就能跑——它是 llama.cpp 生态的标准格式，Ollama、LM Studio 这些本地工具的底层都是它。

GGUF 模型名里那串 `Q4_K_M` 才是真正的信息量，按「比特 _ 结构 _ 档位」三段读：

```text
Q4_K_M
 │  │  └─ M = Medium：部分关键层用更高精度（Q6_K），其余层用 Q4_K
 │  └──── K = K-quant：按 block 组织、带独立 scale 的量化结构
 └─────── 4 = 主权重的比特数
```

K-quant 家族的做法和本章第 3 节的 per-group 完全同源：每 32 个权重一个 block，各有自己的 scale，再按 256 个 block 组成 super-block 进一步压缩 scale 本身。家族成员按「体积-质量」排：`Q2_K`（最小、损失最明显）、`Q3_K_S` / `Q3_K_M`、`Q4_K_S` / `Q4_K_M`、`Q5_K_M`、`Q6_K`、`Q8_0`（接近无损的基线）。经验上 `Q4_K_M` 是体积和质量的常用平衡点，也是社区里下载量最大的档位之一。

再进一步的工具是 `imatrix`（importance matrix）：用一份校准语料统计每个位置权重的重要性，让量化误差往不重要的位置集中——思路和 AWQ 保护重要通道同源。llama.cpp 的 `llama-imatrix` 工具离线产出，量化时挂上即可。

什么时候选 GGUF：没有独立 GPU 的机器（纯 CPU / Apple Silicon）、显存很小、端侧设备，或者只想在笔记本上快速跑一个模型。GPU 服务器上的高并发服务，vLLM / SGLang 的 GPTQ / AWQ / FP8 生态仍是主流——两边不是竞争关系，各自绑定自己的运行时。

## 9. 量化方案的选择

把部署中实际会遇到的选择放进一张表。逻辑就两步：先看你在哪类硬件上跑（这决定运行时），再看精度预算（这决定格式）：

| 部署目标 | 常见格式 | 运行时 / 工具链 |
|:---|:---|:---|
| NVIDIA GPU 高并发服务 | GPTQ / AWQ（INT4）、FP8 checkpoint | vLLM / SGLang / TensorRT-LLM |
| Blackwell 新硬件 | NVFP4 / MXFP4 | TensorRT-LLM / vLLM |
| 快速实验、显存紧张 | bitsandbytes nf4（`load_in_4bit`） | transformers |
| CPU / Mac / 端侧 | GGUF Q4_K_M 等 | llama.cpp / Ollama / LM Studio |
| 长上下文高并发 | W8A8 或 FP8 权重 + FP8 KV | vLLM / SGLang |

选型时真正决定成败的一步是**查引擎的支持矩阵**：同一个「4-bit」，GPTQ 的 checkpoint 拿到 llama.cpp 里跑不了，GGUF 也进不了 vLLM 的主路径——格式和运行时是绑定的。先定引擎，再挑格式，最后才轮到比特数。

## 10. 量化实战

前九节都在讲原理。这一节把最常见的三条量化路径各走一遍：产出 vLLM 能跑的 GPTQ / FP8、AWQ，以及产出 llama.cpp 能跑的 GGUF。命令都需要 GPU（量化过程要跑校准数据），建议放在独立终端执行；先记住三条路径的全景：

| 路径 | 工具 | 产出 | 给谁跑 |
|:---|:---|:---|:---|
| GPTQ / FP8 | llm-compressor | HF 目录 checkpoint | vLLM / SGLang |
| AWQ | AutoAWQ | HF 目录 checkpoint | vLLM / SGLang |
| GGUF | llama.cpp 两件套 | 单文件 `.gguf` | llama.cpp / Ollama |

### 路径一 GPTQ 与 FP8（llm-compressor）

llm-compressor 是 vLLM 生态的官方量化工具，GPTQ 和 FP8 共用同一套 `oneshot` 接口，只换 recipe。GPTQ 版：

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor.transformers import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

model_id = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# W4A16 就是第 4 节的记号；lm_head 对精度最敏感，通常跳过不压
recipe = GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
oneshot(model=model, tokenizer=tokenizer, recipe=recipe,
        output_dir="Qwen2.5-7B-Instruct-GPTQ-W4A16")
```

校准数据默认从模型自身采样，7B 量化大约需要一张 24 GB 卡跑几十分钟。把 recipe 换一行就是 FP8：

```python
from llmcompressor.modifiers.quantization import QuantizationModifier

# FP8_DYNAMIC：权重离线量化，activation 用动态 scale（运行时统计）
recipe = QuantizationModifier(targets="Linear", scheme="FP8_DYNAMIC",
                              ignore=["lm_head"])
oneshot(model=model, tokenizer=tokenizer, recipe=recipe,
        output_dir="Qwen2.5-7B-Instruct-FP8")
```

`FP8_DYNAMIC` 正是第 6 节「activation 随输入变化、难以离线量化」的工程答案——activation 的 scale 不在离线阶段定死，运行时按张量统计。

### 路径二 AWQ（AutoAWQ）

```python
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_path = "Qwen/Qwen2.5-7B-Instruct"
model = AutoAWQForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4,
                "version": "GEMM"}
model.quantize(tokenizer, quant_config=quant_config)

model.save_quantized("Qwen2.5-7B-Instruct-AWQ")
tokenizer.save_pretrained("Qwen2.5-7B-Instruct-AWQ")
```

`quant_config` 里的参数全部是前文出现过的概念：`w_bit=4` 是比特数，`q_group_size=128` 是第 3 节的组粒度，`zero_point=True` 是非对称量化。`quantize` 默认用内置校准集统计各通道的重要性（第 5 节 AWQ 的核心步骤），也可以通过 `calib_data` 换成自己场景的语料——校准数据越贴近真实流量，保护的重要通道越准。

### 路径三 GGUF（llama.cpp）

GGUF 的制作是两步：先把 HF 权重转成未量化的 F16 GGUF，再压到目标档位。中间产物保留下来，还能压其他档位做对比。

```bash
git clone https://github.com/ggml-org/llama.cpp
pip install -r llama.cpp/requirements.txt

# 第一步：HF 权重 -> F16 GGUF
python llama.cpp/convert_hf_to_gguf.py /path/to/Qwen2.5-7B-Instruct \
  --outfile qwen2.5-7b-f16.gguf

# 第二步：量化到第 8 节拆过的 Q4_K_M
./llama-quantize qwen2.5-7b-f16.gguf qwen2.5-7b-q4_k_m.gguf Q4_K_M
```

要挂第 8 节提到的 imatrix，中间加一步统计、量化时挂上：

```bash
# 用一份校准文本统计重要性
./llama-imatrix -m qwen2.5-7b-f16.gguf -f calibration.txt -o imatrix.dat
./llama-quantize qwen2.5-7b-f16.gguf qwen2.5-7b-q4_k_m.gguf Q4_K_M \
  --imatrix imatrix.dat
```

### 跑起来

产出之后，启动只是一行命令（完整流程见「模型部署与服务化」一本）：

```bash
# GPTQ / AWQ checkpoint 会被自动识别
vllm serve ./Qwen2.5-7B-Instruct-AWQ

# 在线压 FP8：BF16 checkpoint 加载时直接压，省掉离线步骤
vllm serve Qwen/Qwen2.5-7B-Instruct --quantization fp8

# llama.cpp 同样暴露 OpenAI-compatible API
llama-server -m qwen2.5-7b-q4_k_m.gguf --port 8000
```

第二条值得单独记：`--quantization fp8` 允许把现成的 BF16 模型在加载时在线压成 FP8，不用预先做离线量化——适合快速试验。正式上线仍建议用离线校准过的 checkpoint：校准数据贴近真实流量，精度更有保障。

## 11. 量化的收益与代价

量化不是免费的午餐，把两边的账摆在一起看。

**收益**：显存直接缩小（INT4 理论 4 倍）；Decode 是 memory-bound，每步搬运的数据变少，吞吐随之上升；省出的显存可以换更大的模型、更长的上下文或更高的并发；新硬件上 FP8 / FP4 还有原生加速。

**代价**：质量下降。典型表现是困惑度微升，数学、代码这类对精度敏感的任务掉分更明显；而且**模型越小越敏感**——经验上，30B 以上的模型压 4-bit 往往接近无损，7B 到 14B 用 GPTQ / AWQ 的 4-bit 通常可接受，3B 以下就要谨慎，优先考虑 8-bit 或 FP8。此外 per-group scale 增加元数据、部分 kernel 不支持混合精度需要先反量化，都是小的工程成本。

所以上线前的标准动作是：按第 9 节的表选格式、按第 10 节的命令压一版，再按评测一本的流程跑一轮质量对比——「INT4 省下的显存，值不值得这一点分数」，只有你自己场景下的 benchmark 能回答。

## 小结

这一章把「低比特」拆成了可以分开回答的问题：

- 压谁？权重（W）、activation（A）、KV Cache——三个不同对象
- 压成什么？INT4 / INT8 的均匀格子，或 FP8 / FP4 的浮点格子——后者贴合「挤在 0 附近」的分布，新硬件原生支持
- 粒度多细？tensor / channel / group——outlier 决定了不能太粗
- 误差怎么救？RTN 是底线；GPTQ 补误差、AWQ 保通道、SmoothQuant 挪难度、QAT 直接训
- 选什么格式？先定运行时再挑格式：GPU 服务走 GPTQ / AWQ / FP8，端侧走 GGUF（Q4_K_M 一族），快速实验走 bitsandbytes
- 怎么做？llm-compressor 出 GPTQ / FP8，AutoAWQ 出 AWQ，llama.cpp 两步出 GGUF；vLLM 还支持加载时在线压 FP8
- 划不划算？小模型更敏感；上线前用评测一本的流程验证质量

下一章继续沿着 Decode 的串行瓶颈走：

> **每一步便宜了，但一次 forward 还是只能确认一个 Token。能不能一次确认多个？**

## 作业

三道题分别对应：粒度、非对称量化、读模型名——实际部署时最先碰到的三件事。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。

### 作业 1：实现 per-group 量化并比较误差

逐行把每 `group` 个相邻元素用组内 scale 量化再还原，然后和 per-tensor 比误差。

**小提示**：组内 scale 是 `np.max(np.abs(block)) / 7`，之后 round、clip 到 `[-7, 7]` 再乘回。

In [ ]:
# 作业 1：per-group 量化 填空

np.random.seed(0)
W_test = np.random.randn(8, 32).astype(np.float32)
W_test[0] *= 10  # 制造一个 outlier 行

def qdq_group_cols(a, group=8):
    """逐行把每 group 个相邻元素用组内 scale 量化再还原"""
    out = np.zeros_like(a)
    for r in range(a.shape[0]):
        for c0 in range(0, a.shape[1], group):
            block = a[r, c0:c0 + group]
            # TODO：把下面三引号里的内容替换成你的代码
            """算组内 scale，round/clip 后乘回，写进 out[r, c0:c0+group]"""
    return out

err_tensor = np.mean(np.abs(W_test - qdq_tensor(W_test)))
err_group = np.mean(np.abs(W_test - qdq_group_cols(W_test)))
assert err_group < err_tensor, (err_group, err_tensor)
print("✅ 作业 1 通过：组内 outlier 只污染自己那组，误差更小")

### 作业 2：非对称量化（zero point）

数值几乎全在正半轴时（很多 Activation 就是这样），对称量化会浪费一半格子。
非对称量化把 `[min, max]` 整段映射到 `[0, 14]`，再记一个 zero point 标出原点位置。

**小提示**：`scale = (max - min) / 14`，`zero_point = round(-min / scale)`；
量化 `q = round(x / scale) + zero_point` 后 clip 到 `[0, 14]`，反量化 `(q - zero_point) * scale`。

In [ ]:
# 作业 2：非对称量化 填空

x = np.array([2.0, 2.5, 3.0, 3.5, 4.0], dtype=np.float32)  # 全在正半轴
scale = (x.max() - x.min()) / 14
zero_point = int(np.round(-x.min() / scale))

def qdq_asym(a):
    """把 [min, max] 映射到 [0, 14] 的量化-反量化"""
    # TODO：把下面三引号里的内容替换成你的代码
    """q = round(a/scale) + zero_point 后 clip 到 [0,14]，再反量化回来"""

x_hat = qdq_asym(x)
assert np.max(np.abs(x - x_hat)) <= scale / 2 + 1e-6, (x, x_hat)
print("✅ 作业 2 通过：非对称量化把整段范围用满，误差不超过半个格子")

### 作业 3：读懂量化模型的名字

Hugging Face 上的名字像 `Qwen2.5-7B-Instruct-GPTQ-Int4`，要能拆出方法和精度。

**小提示**：`Int8` 出现就是 8-bit，否则默认 4-bit；方法在 `GPTQ` / `AWQ` / `GGUF` 里找。

In [ ]:
# 作业 3：模型名拆解 填空

def parse_quant_name(name):
    """从量化模型名拆出 {'bits': int, 'method': str}"""
    # TODO：把下面三引号里的内容替换成你的代码
    """bits 取 Int8/Int4 中的数字（默认 4）；method 取 GPTQ/AWQ/GGUF 之一"""

info = parse_quant_name("Qwen2.5-7B-Instruct-GPTQ-Int4")
assert info == {"bits": 4, "method": "GPTQ"}, info
info2 = parse_quant_name("Llama-3-8B-AWQ")
assert info2 == {"bits": 4, "method": "AWQ"}, info2
print("✅ 作业 3 通过：看到量化模型名，你能马上说出方法和精度")